# QuantJourney SDK - Domain Route Discovery and Contract Introspection

This notebook demonstrates a QuantJourney SDK workflow that uses route descriptions and a concrete AAPL data packet to show what a governed domain route returns for research.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


In [ ]:
route_names = ['equity.pricing.get_historical_prices', 'equity.fundamentals.get_financial_ratios_ttm', 'reference.identifiers.get_figi_data']
descriptions = {route: qj.domains.describe(route=route) for route in route_names}
prices_raw = qj.eod.get_historical_prices(symbol='AAPL', start_date='2024-01-01', end_date=END)
ratios_raw = qj.fmp.get_financial_ratios_ttm(symbol='AAPL')
identity_raw = qj.openfigi.get_figi_data(symbol='AAPL', exchange='US')


In [ ]:
rows = []
for route, payload in descriptions.items():
    value = unwrap(payload)
    if isinstance(value, dict):
        rows.append({'route': route, 'domain': value.get('domain') or value.get('domain_path'), 'description': value.get('description'), 'required_scopes': value.get('required_scopes') or value.get('scopes'), 'connectors': value.get('connectors') or value.get('providers')})
    else:
        rows.append({'route': route, 'domain': None, 'description': None, 'required_scopes': None, 'connectors': None})
contract = pd.DataFrame(rows)
display(contract)


In [ ]:
prices = pd.DataFrame(as_rows(prices_raw))
if prices.empty:
    raise RuntimeError('No AAPL price rows returned')
prices['date'] = pd.to_datetime(prices['date'], errors='coerce')
prices['price'] = pd.to_numeric(prices.get('adjusted_close', prices.get('close')), errors='coerce')
prices = prices.dropna(subset=['date', 'price']).set_index('date').sort_index()
ratios = pd.DataFrame(as_rows(ratios_raw))
identity = pd.DataFrame(as_rows(identity_raw))
display(ratios.head())
display(identity.head())
prices['price'].tail(252).plot(title='AAPL adjusted price returned through the route packet')
plt.ylabel('price')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.